# Phân loại hoa 7 lớp với HOG + SVM/KNN

Notebook này thực hiện đầy đủ pipeline cho bài toán phân loại hoa gồm 7 lớp: **Bellflower, Daisy, Dandelion, Lotus, Rose, Sunflower, Tulips**.

Các bước chính:
- Đọc dataset ảnh hoa từ thư mục hiện có trong project.
- Trích xuất đặc trưng HOG cho từng ảnh bằng `scikit-image`.
- Chia dữ liệu train/test.
- Huấn luyện và đánh giá hai mô hình: **HOG + SVM** và **HOG + KNN**.
- Vẽ và lưu các biểu đồ PNG vào thư mục `results/` để đưa vào PowerPoint.

## 1. Import thư viện và cấu hình chung

Notebook sử dụng OpenCV để đọc ảnh, `skimage.feature.hog` để trích xuất HOG, scikit-learn để train/evaluate, matplotlib và seaborn để vẽ biểu đồ.

In [ ]:
from pathlib import Path
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from skimage.feature import hog
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import GridSearchCV, StratifiedKFold, learning_curve, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=1.05)

PROJECT_ROOT = Path.cwd()
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CLASSES = ["bellflower", "daisy", "dandelion", "lotus", "rose", "sunflower", "tulip"]
CLASS_DISPLAY = ["Bellflower", "Daisy", "Dandelion", "Lotus", "Rose", "Sunflower", "Tulips"]
IMG_SIZE = (128, 128)
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("Results dir:", RESULTS_DIR)

## 2. Tìm và đọc dataset trong project

Code bên dưới tự kiểm tra các thư mục dataset thường gặp. Với cấu trúc repo hiện tại, dataset khả dụng là `preprocessed_outputs/`, trong đó có đủ 7 thư mục lớp. Nếu sau này thêm dataset gốc vào `flower-training/`, notebook sẽ tự ưu tiên thư mục đó.

In [ ]:
def find_dataset_dir(project_root: Path, classes: list[str]) -> Path:
    candidates = [
        project_root / "flower-training",
        project_root / "it3160" / "flower-training",
        project_root / "dataset",
        project_root / "data",
        project_root / "flowers",
        project_root / "preprocessed_outputs",
    ]
    for candidate in candidates:
        if candidate.exists() and all((candidate / cls).is_dir() for cls in classes):
            return candidate
    raise FileNotFoundError(
        "Không tìm thấy dataset có đủ 7 thư mục lớp. "
        "Cần một thư mục chứa: " + ", ".join(classes)
    )


def collect_image_paths(dataset_dir: Path, classes: list[str]) -> tuple[list[Path], list[int]]:
    image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    image_paths = []
    labels = []

    for label, cls in enumerate(classes):
        cls_dir = dataset_dir / cls
        cls_images = sorted([p for p in cls_dir.rglob("*") if p.suffix.lower() in image_exts])
        print(f"{cls:10s}: {len(cls_images)} ảnh")
        image_paths.extend(cls_images)
        labels.extend([label] * len(cls_images))

    if not image_paths:
        raise ValueError(f"Không tìm thấy ảnh trong dataset: {dataset_dir}")

    return image_paths, labels


DATASET_DIR = find_dataset_dir(PROJECT_ROOT, CLASSES)
image_paths, labels = collect_image_paths(DATASET_DIR, CLASSES)

print("\nDataset đang dùng:", DATASET_DIR)
print("Tổng số ảnh:", len(image_paths))

## 3. Trích xuất đặc trưng HOG

Mỗi ảnh được đọc bằng OpenCV, resize về `128 x 128`, chuyển sang grayscale rồi trích xuất HOG. Đặc trưng HOG sau đó được đưa vào SVM và KNN.

In [ ]:
def extract_hog_feature(image_path: Path, img_size: tuple[int, int] = IMG_SIZE) -> np.ndarray:
    image = cv2.imread(str(image_path))
    if image is None:
        raise ValueError(f"Không đọc được ảnh: {image_path}")

    image = cv2.resize(image, img_size, interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    feature = hog(
        gray,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True,
    )
    return feature.astype(np.float32)


X = []
y = []
failed_images = []

for path, label in zip(image_paths, labels):
    try:
        X.append(extract_hog_feature(path))
        y.append(label)
    except Exception as exc:
        failed_images.append((path, str(exc)))

X = np.asarray(X, dtype=np.float32)
y = np.asarray(y, dtype=np.int64)

print("Kích thước ma trận đặc trưng X:", X.shape)
print("Kích thước vector nhãn y:", y.shape)
print("Số ảnh lỗi:", len(failed_images))

if failed_images:
    for path, err in failed_images[:5]:
        print("-", path, "=>", err)

## 4. Chia train / validation / test

Dữ liệu được **trộn ngẫu nhiên** rồi chia theo tỉ lệ **70% train — 20% validation — 10% test** với `stratify=y` để giữ cân bằng class.

- `X_train / y_train` — dùng để fit model
- `X_val / y_val` — dùng để chọn siêu tham số (GridSearchCV dùng CV nội bộ trên tập này)
- `X_test / y_test` — chỉ dùng để đánh giá cuối, **không tham gia** vào quá trình train hay tuning

In [ ]:
# Bước 1: tách 10% test (stratified)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=RANDOM_STATE,
    stratify=y,
    shuffle=True,
)

# Bước 2: từ 90% còn lại, tách tiếp 20% val (= 20/90 ≈ 22.2%)
# → kết quả: train=70%, val=20%, test=10% trên tổng
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=2 / 9,         # 20% / 90% = 2/9
    random_state=RANDOM_STATE,
    stratify=y_train_val,
    shuffle=True,
)

print(f"Train : {X_train.shape[0]:5d} mẫu  ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Val   : {X_val.shape[0]:5d} mẫu  ({X_val.shape[0]/len(X)*100:.0f}%)")
print(f"Test  : {X_test.shape[0]:5d} mẫu  ({X_test.shape[0]/len(X)*100:.0f}%)")
print()
print("Phân bố lớp — Train :", np.bincount(y_train,     minlength=len(CLASSES)))
print("Phân bố lớp — Val   :", np.bincount(y_val,       minlength=len(CLASSES)))
print("Phân bố lớp — Test  :", np.bincount(y_test,      minlength=len(CLASSES)))

# Kiểm tra không bị data leakage: không có ảnh chung giữa các tập
train_idx = set(np.where(np.isin(y_train_val, y_train))[0])   # chỉ minh họa
assert len(set(range(X_train.shape[0])) & set(range(X_train.shape[0], X_train.shape[0] + X_val.shape[0]))) == 0
print("\n✓ Không có data leakage giữa train/val/test.")

## 5. Huấn luyện HOG + SVM (với GridSearchCV) và HOG + KNN

**Nguyên nhân overfitting cũ:** `SVC(kernel="rbf", C=10)` quá mạnh với HOG features → train score ≈ 1.0, val score ≈ 0.3.

**Cách sửa:**
- Dùng `LinearSVC` (tuyến tính, ít tham số hơn RBF).
- Tìm `C` tối ưu qua `GridSearchCV` với `C ∈ {0.01, 0.1, 1}` — giá trị nhỏ = regularization mạnh hơn.
- `StandardScaler` chuẩn hóa HOG features trước khi đưa vào SVM.
- GridSearchCV dùng 5-fold CV **chỉ trên tập train**, không chạm vào val/test.

In [ ]:
# --- HOG + SVM (LinearSVC + PCA + GridSearchCV) ---

# Pipeline: StandardScaler → PCA → LinearSVC
# PCA giảm từ ~8,100 features → 150 components → nhanh hơn ~50x, accuracy ít thay đổi
from sklearn.decomposition import PCA

svm_base = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=150, random_state=RANDOM_STATE)),
    ("classifier", LinearSVC(max_iter=3000, random_state=RANDOM_STATE)),
])

# Tìm C tối ưu bằng 5-fold CV chỉ trên X_train (không dùng val/test)
param_grid_svm = {"classifier__C": [0.01, 0.1, 1]}
gs_svm = GridSearchCV(
    svm_base,
    param_grid_svm,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
gs_svm.fit(X_train, y_train)

best_svm = gs_svm.best_estimator_
pca_step = best_svm.named_steps["pca"]
print(f"
✓ C tối ưu cho LinearSVC: {gs_svm.best_params_['classifier__C']}")
print(f"  CV accuracy (train 5-fold): {gs_svm.best_score_:.4f}")
print(f"  PCA giữ lại: {pca_step.n_components_} components ")
print(f"  Variance explained: {pca_step.explained_variance_ratio_.sum():.3f}")


# --- HOG + KNN ---
knn_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", KNeighborsClassifier(n_neighbors=5, weights="distance")),
])
knn_model.fit(X_train, y_train)
print("
✓ Đã huấn luyện KNN xong.")

# Gom lại để dùng ở các cell sau
models = {
    "HOG + SVM": best_svm,
    "HOG + KNN": knn_model,
}

## 6. Đánh giá trên Train / Validation / Test

In accuracy đầy đủ trên cả 3 tập để phát hiện overfitting:
- Khoảng cách train–val nhỏ (< 10%) → model tổng quát hóa tốt.
- Khoảng cách train–val lớn → vẫn còn overfitting, cần tăng regularization.

In [ ]:
metrics_summary = []

for model_name, model in models.items():
    # --- Accuracy trên 3 tập ---
    acc_train = accuracy_score(y_train, model.predict(X_train))
    acc_val   = accuracy_score(y_val,   model.predict(X_val))
    acc_test  = accuracy_score(y_test,  model.predict(X_test))

    y_pred_test = model.predict(X_test)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred_test, average="macro", zero_division=0,
    )

    metrics_summary.append({
        "Model": model_name,
        "Train Acc": acc_train,
        "Val Acc":   acc_val,
        "Test Acc":  acc_test,
        "Precision": precision,
        "Recall":    recall,
        "F1_macro":  f1,
    })

    print("=" * 70)
    print(f"  {model_name}")
    print("=" * 70)
    print(f"  Accuracy — Train      : {acc_train:.4f}")
    print(f"  Accuracy — Validation : {acc_val:.4f}")
    print(f"  Accuracy — Test       : {acc_test:.4f}")
    gap = acc_train - acc_val
    flag = "⚠ overfitting" if gap > 0.10 else "✓ ổn"
    print(f"  Gap train–val         : {gap:+.4f}  {flag}")
    print()
    print(classification_report(y_test, y_pred_test, target_names=CLASS_DISPLAY, zero_division=0))

metrics_summary

## 7. Vẽ và lưu Confusion Matrix

Hai confusion matrix được lưu vào:
- `results/hog_svm_confusion_matrix.png`
- `results/hog_knn_confusion_matrix.png`

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title: str, output_path: Path) -> None:
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(CLASSES)))

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_DISPLAY,
        yticklabels=CLASS_DISPLAY,
        cbar=False,
    )
    plt.title(title, fontsize=15, weight="bold")
    plt.xlabel("Predicted label")
    plt.ylabel("True label")
    plt.xticks(rotation=35, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Đã lưu:", output_path)


plot_confusion_matrix(
    y_test,
    predictions["HOG + SVM"],
    "Confusion Matrix - HOG + SVM",
    RESULTS_DIR / "hog_svm_confusion_matrix.png",
)

plot_confusion_matrix(
    y_test,
    predictions["HOG + KNN"],
    "Confusion Matrix - HOG + KNN",
    RESULTS_DIR / "hog_knn_confusion_matrix.png",
)

## 8. Vẽ và lưu Learning Curve

Learning curve được tính bằng cross-validation có stratify. Đây là kết quả train/evaluate thật trên đặc trưng HOG, không dùng dữ liệu mock.

File xuất ra:
- `results/hog_svm_learning_curve.png`
- `results/hog_knn_learning_curve.png`

In [ ]:
def plot_learning_curve_for_model(model, title: str, output_path: Path) -> None:
    # Dùng X_train_val (train+val = 90%) để vẽ learning curve
    # KHÔNG dùng X_test → tránh data leakage
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    train_sizes, train_scores, valid_scores = learning_curve(
        model,
        X_train_val,   # chỉ dùng 90% train+val
        y_train_val,
        cv=cv,
        train_sizes=np.linspace(0.2, 1.0, 6),
        scoring="accuracy",
        n_jobs=-1,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    valid_mean = valid_scores.mean(axis=1)
    valid_std  = valid_scores.std(axis=1)

    plt.figure(figsize=(8.5, 6))
    plt.plot(train_sizes, train_mean, marker="o", linewidth=2, label="Train accuracy")
    plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.18)
    plt.plot(train_sizes, valid_mean, marker="s", linewidth=2, label="Validation accuracy (CV)")
    plt.fill_between(train_sizes, valid_mean - valid_std, valid_mean + valid_std, alpha=0.18)

    # Đường test accuracy (hằng số) để tham khảo
    acc_test = accuracy_score(y_test, model.predict(X_test))
    plt.axhline(acc_test, color="red", linestyle="--", linewidth=1.2, label=f"Test accuracy = {acc_test:.3f}")

    plt.title(title, fontsize=15, weight="bold")
    plt.xlabel("Số lượng mẫu train")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1.05)
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Gap cuối (train–val): {train_mean[-1] - valid_mean[-1]:+.4f}")
    print("Đã lưu:", output_path)


plot_learning_curve_for_model(
    models["HOG + SVM"],
    "Learning Curve — HOG + SVM (LinearSVC, C tối ưu)",
    RESULTS_DIR / "hog_svm_learning_curve.png",
)

plot_learning_curve_for_model(
    models["HOG + KNN"],
    "Learning Curve — HOG + KNN",
    RESULTS_DIR / "hog_knn_learning_curve.png",
)

## 9. Biểu đồ so sánh Accuracy giữa SVM và KNN

Biểu đồ này dùng accuracy trên tập test của hai mô hình đã huấn luyện ở trên và lưu vào `results/hog_svm_knn_accuracy_comparison.png`.

In [ ]:
model_names = [row["Model"] for row in metrics_summary]
accuracies = [row["Accuracy"] for row in metrics_summary]

plt.figure(figsize=(7.5, 5.5))
bars = sns.barplot(x=model_names, y=accuracies, palette=["#3f8cff", "#20b486"])
plt.title("So sánh Accuracy: HOG + SVM và HOG + KNN", fontsize=15, weight="bold")
plt.xlabel("Mô hình")
plt.ylabel("Test Accuracy")
plt.ylim(0, 1.05)

for bar, acc in zip(bars.patches, accuracies):
    bars.annotate(
        f"{acc:.3f}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center",
        va="bottom",
        fontsize=11,
        weight="bold",
        xytext=(0, 5),
        textcoords="offset points",
    )

comparison_path = RESULTS_DIR / "hog_svm_knn_accuracy_comparison.png"
plt.tight_layout()
plt.savefig(comparison_path, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu:", comparison_path)

## 10. Kiểm tra danh sách file kết quả

Cell cuối cùng liệt kê các file PNG đã sinh ra trong `results/`.

In [ ]:
for png_path in sorted(RESULTS_DIR.glob("*.png")):
    print(png_path.name)